# P1 수심 계약 v2 — 재사용 가능한 학습·내부검증 기록

## tl;dr

동일 421,032행의 development 정책 순위는 `control`가 가장 높았다. 기준 control 대비 ΔF1=+0.00000000. 공식 점수 상승은 미측정이며, 공식 입력/CSV/업로드는 이 사이클에서 0이다.

## Context & Methods

A: 현재 depth를 2m 반올림하는 연도 독립 특징, 나머지 recipe 동일. Q2/Q3/Q4 purge21일·earlier-inner60일·seed20260813, 신규12 historical+2 final-inner+2 full fits. B: legacy OOF 출처/키만 감사, 119행 fold 불일치 및 router purge7일로 무학습 결합 차단. C: A/B 완료 뒤 고정 OFF/ON(lambda1), 0 backbone fits.

### Key Assumptions

반복 노출된 historical development다. 월별 악화/CI는 위험 근거이며 새 hard gate가 아니다. 이 노트북은 집계 QA의 실행 가능한 companion으로 학습을 다시 시작하거나 공식 값을 읽지 않는다.

### 학습 실행 경로

새 isolated checkout에서 `P1_DATA_DIR`를 배포 데이터 폴더로 설정하고 `.venv-p1/Scripts/python.exe scripts/run_p1_depth_contract_repair_20260905_v2.py --execute`를 실행한다. 기존 artifact가 있는 checkout에서는 one-shot 보호로 재실행되지 않는다. 완료 후 같은 runner `--verify`, postaudit `--provenance`(1회), `--decoder`, 독립 QA runner 순서다. 모델/config/lock를 지우거나 수정하여 재실행하지 않는다. 04_models의 저장 모델만 추론하는 것과 위 학습 절차를 구분한다.

## Data

배포 train 776,706행의 실행 영수증과 OOF 집계만 읽는다. 원시 관측·행별 정답은 이 노트북에 없다.

In [1]:
import json
from pathlib import Path
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs').is_dir() and (p / 'scripts').is_dir())
report_dir = project_root / 'reports/p1_depth_contract_repair_20260905_v2'
read = lambda name: json.loads((report_dir / name).read_text(encoding='utf-8'))
training = read('result.json')
decoder = read('decoder-result.json')
audit = read('cycle-independent-qa.json')
assert audit['status'] == 'PASS'
{key: training[key] for key in ['status', 'train_sha256', 'screen_fits', 'final_inner_fits', 'full_fits', 'runtime_seconds', 'official_rows']}

{'status': 'TERMINAL_MODELS_FROZEN_OFFICIAL_UNREAD',
 'train_sha256': '20b656b0cbd524ad9da0bae8ecb6e0bacfc006e05810b37e83f29a5fa8e65cd2',
 'screen_fits': 12,
 'final_inner_fits': 2,
 'full_fits': 2,
 'runtime_seconds': 765.2190000000119,
 'official_rows': 0}

## Results

### 동일 평가키의 주요 지표

F1은 fold 평균이 아니라 pool한 TP/FP/FN에서 산출한다.

In [2]:
[{ 'policy': name, **values } for name, values in audit['independent_metrics'].items()]

[{'policy': 'control',
  'f1': 0.8511742399041979,
  'tp': 12794,
  'fp': 1213,
  'fn': 3261},
 {'policy': 'candidate',
  'f1': 0.8489614440241555,
  'tp': 12793,
  'fp': 1290,
  'fn': 3262},
 {'policy': 'candidate_decoder_on',
  'f1': 0.8498715853373804,
  'tp': 12740,
  'fp': 1186,
  'fn': 3315}]

### 불확실성과 분할 계약

In [3]:
provenance = read('provenance-audit.json')
{'intervals': decoder['intervals_vs_control'], 'B_status': provenance['status'], 'fold_changed_rows': provenance['changed_fold_rows'], 'QA': [audit['passed'], audit['check_count']]}

{'intervals': {'candidate': {'unit': '7-day shared station blocks within historical fold',
   'blocks': 38,
   'replicates': 2000,
   'seed': 20260905,
   'ci90': [-0.006314360617607073, 0.0023703576410936673],
   'role': 'retrospective development uncertainty, not fresh confirmation or hard gate'},
  'candidate_decoder_on': {'unit': '7-day shared station blocks within historical fold',
   'blocks': 38,
   'replicates': 2000,
   'seed': 20260905,
   'ci90': [-0.005435345019545468, 0.002958016972807381],
   'role': 'retrospective development uncertainty, not fresh confirmation or hard gate'}},
 'B_status': 'B_BLOCKED_EXACT_CONTRACT_MISMATCH_NOT_SCIENTIFIC_NO_GO',
 'fold_changed_rows': 119,
 'QA': [24, 24]}

## Takeaways

정책의 실제 효과와 수심 결측 계약의 수정 여부를 구분한다. B의 미실행은 성능 실패가 아니다. 공식 예상 점수는 미산정이며 마지막 소수의 Public 반환값으로 환산식을 맞추지 않는다. 자세한 fold/월/정점층 결과는 decoder-result.json과 report-source.md에 보존한다. 현재 모델은 연구 자산이며 기존 최종 제출 패키지를 대체하지 않았다.